# Unified re-runs for Tables 1 and 4, and the release checkpoint

Earlier tables were produced in separate notebooks, each drawing its own AIOMFAC labels, which led to small
inconsistencies (e.g. R_emb on BIMOG: 0.161 in Table 1 vs 0.147 in Table 2). Here every model is trained on the
**same label set and protocol** as `representation_C_reruns.ipynb`, which produced Tables 2, 4 (R_emb rows)
and 7:

- **Table 1** (representation comparison): hash-split columns use the Table 2 protocol (fixed interpolation /
  validation split, ≤ 500 epochs), and the CHO cross-validation uses the same labels.
- **Table 4** (learning curves): R_desc MLP and GNN rows are re-trained on the same MCM labels and subsets as the
  R_emb rows.
- **Release checkpoint**: an R_emb ensemble (3 seeds) trained on all BIMOG + MCM molecules, saved with its
  vocabulary and normalization statistics.

In [1]:
import json, time, hashlib
import numpy as np, pandas as pd, torch, torch.nn as nn
from scipy import stats
from rdkit import Chem, RDLogger
from aiomfac_py import ActivityModel, Component
from aiomfac_py.s2as import smiles_to_components
from aiomfac_py.params import load_subgroup_params
import aiomfac_py.s2as._mapping as _m
_m.print = lambda *a, **k: None; RDLogger.DisableLog("rdApp.*")
sg = load_subgroup_params(); WATER = Component(1, "Water", ((16, 1),))
hash100 = lambda s: int(hashlib.sha256(s.encode()).hexdigest(), 16) % 100
fmt = lambda s: f"{np.mean(s):.3f} ± {np.std(s, ddof=1):.3f}" if len(s) > 1 else f"{np.mean(s):.3f}"
T0 = time.time()

def featurize(name, smi, cat):
    r = smiles_to_components([smi], water_as_component1=True)
    if r.removed: return None
    try:
        if not (ActivityModel(r.components).evaluate([0.9, 0.1], 298.15, basis="mass").gamma_neutral > 0).all(): return None
    except Exception: return None
    mol = Chem.MolFromSmiles(smi); mh = Chem.AddHs(mol); sym = [a.GetSymbol() for a in mh.GetAtoms()]; nc = max(sym.count("C"), 1)
    return {"name": name, "smiles": smi, "canon": Chem.MolToSmiles(mol), "cat": cat, "subgroups": r.components[1].subgroups,
            "M": sum(a.GetMass() for a in mh.GetAtoms()), "oc": sym.count("O") / nc, "nc": sym.count("N") / nc,
            "hc": sym.count("H") / nc, "unm": int(r.n_unmatched_atoms[0])}

def bimog_cat(d):
    n, h = d["#N"] != 0, d["#Hal"] != 0
    return "n_and_hal" if n and h else "n_only" if n else "hal_only" if h else "cho"

bim_raw = json.load(open("bimog_unified.json"))
bimog = [o for d in bim_raw if (o := featurize(d["name"], d["smiles"], bimog_cat(d)))]
bcanon = {o["canon"] for o in bimog}

synth = []
add = lambda s, n: synth.append((n, s))
for n in range(1, 11): add("C" * n + "N", f"amine1-C{n}")
for n in range(1, 7): add("C" * n + "N" + "C" * n, f"amine2s-C{n}")
for n in range(1, 6):
    for m in range(n + 1, 7): add("C" * n + "N" + "C" * m, f"amine2-C{n}C{m}")
for n in range(1, 9): add("C" * n + "[N+](=O)[O-]", f"nitro-C{n}")
add("c1ccccc1[N+](=O)[O-]", "nitrobenzene")
for n in range(1, 5): add("C" * n + "c1ccccc1[N+](=O)[O-]", f"nitroarom-C{n}")
for n in range(1, 11): add("C" * n + "O[N+](=O)[O-]", f"nitrate-C{n}")
for n in range(2, 8):
    add("C" * (n - 1) + "C(O)O[N+](=O)[O-]", f"hn-a-C{n}"); add("OC" + "C" * (n - 2) + "CO[N+](=O)[O-]", f"hn-b-C{n}")
for n in range(2, 8): add("NC" + "C" * (n - 2) + "CO", f"aminoalc-C{n}")
for s, n in [("CC(C)N", "isopropylamine"), ("CC(C)(C)N", "tBuNH2"), ("CC(C)CN", "iBuNH2"), ("CC(N)CC", "sBuNH2"),
             ("C1CCC(CC1)N", "cyhexNH2"), ("C1CCC(C1)N", "cypentNH2"), ("CC(C)(C)CN", "neopentNH2"), ("COCCN", "MeOEtNH2"),
             ("CCOCCCN", "EtOPrNH2"), ("CC(=O)CCN", "aminobutanone"), ("CCOC(=O)CCN", "Et-aminopropanoate"),
             ("NCCOCCN", "bisaminoethylether"), ("NCCCCN", "putrescine"), ("NCCCCCN", "cadaverine"), ("NCCCCCCN", "HMDA"),
             ("NC(C)CN", "diaminopropane"), ("CC(C)[N+](=O)[O-]", "2-nitropropane"), ("C1CCC(CC1)[N+](=O)[O-]", "nitrocyclohexane"),
             ("CC(C)(C)[N+](=O)[O-]", "tBuNO2"), ("CC(=O)CC[N+](=O)[O-]", "nitrobutanone"), ("COCC[N+](=O)[O-]", "MeO-nitroethane"),
             ("CC(=O)CO[N+](=O)[O-]", "nitrooxypropanone"), ("COCCO[N+](=O)[O-]", "MeOEt-nitrate"), ("CC(C)O[N+](=O)[O-]", "iPr-nitrate"),
             ("C1CCC(CC1)O[N+](=O)[O-]", "cyhex-nitrate"), ("CC(C)NC(C)C", "diisopropylamine"), ("C1CCNCC1", "piperidine"),
             ("C1CCNC1", "pyrrolidine")]:
    add(s, n)
synth = [o for n, s in synth if (o := featurize(n, s, "synth_n"))]

mraw = pd.read_csv("mcm_v331_species.tsv", sep="\t", comment="*"); mcm = []
for _, r in mraw.iterrows():
    mol = Chem.MolFromSmiles(str(r.Smiles))
    if (mol is None or str(r.Excited).lower() == "true" or any(a.GetNumRadicalElectrons() for a in mol.GetAtoms())
            or not {a.GetSymbol() for a in mol.GetAtoms()} <= {"C", "H", "O", "N"} or mol.GetNumHeavyAtoms() < 2
            or "C" not in {a.GetSymbol() for a in mol.GetAtoms()}): continue
    o = featurize(r.Name, str(r.Smiles), "mcm")
    if o and o["unm"] == 0 and o["canon"] not in bcanon: mcm.append(o)

POOL = bimog + synth + mcm
SRC = np.array(["bimog"] * len(bimog) + ["synth"] * len(synth) + ["mcm"] * len(mcm))
CAT = np.array([o["cat"] for o in POOL]); H = np.array([hash100(o["smiles"]) for o in POOL]); TEST = H < 15
VOCAB = sorted({int(s) for o in POOL for s, q in o["subgroups"]})
def counts(o):
    v = np.zeros(len(VOCAB), np.float32)
    for s, q in o["subgroups"]: v[VOCAB.index(int(s))] = q
    return v
CNT = np.array([counts(o) for o in POOL])
print(f"BIMOG {len(bimog)}, synthetic N {len(synth)}, MCM {len(mcm)}; vocabulary {len(VOCAB)} subgroups ({time.time()-T0:.0f}s)")

BIMOG 368, synthetic N 100, MCM 3337; vocabulary 55 subgroups (10s)


In [2]:
def label(ids, n_comp=20, n_temp=4, seed=0):
    '''AIOMFAC labels for the molecules in ids (Part I sampling). Returns molecule index, (w, T_norm), y.'''
    rng = np.random.RandomState(seed); rows = []
    for oi in ids:
        m = ActivityModel([WATER, Component(2, "o", tuple(POOL[oi]["subgroups"]))]); e = np.linspace(0.02, 0.98, n_comp + 1)
        for w in rng.uniform(e[:-1], e[1:]):
            for T in rng.uniform(273.15, 313.15, n_temp):
                try:
                    lg = m.evaluate([w, 1 - w], T, basis="mass").ln_gamma
                    if np.isfinite(lg[:2]).all(): rows.append((oi, w, (T - 293.15) / 20, lg[0], lg[1]))
                except Exception: pass
    a = np.array(rows)
    return a[:, 0].astype(int), a[:, 1:3].astype(np.float32), a[:, 3:5].astype(np.float32)

PM, EXTRA, Y = label(np.arange(len(POOL)))                     # 20 x 4 labels for every molecule
print(f"{len(Y)} labelled points ({time.time()-T0:.0f}s)")


class EmbedNet(nn.Module):
    def __init__(self, n_vocab, n_extra, n_out=2, emb=16):
        super().__init__()
        self.n_vocab = n_vocab; self.emb = nn.Linear(n_vocab, emb, bias=False)
        self.head = nn.Sequential(nn.Linear(emb + n_extra, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(),
                                  nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, n_out))
    def forward(self, x): return self.head(torch.cat([self.emb(x[:, :self.n_vocab]), x[:, self.n_vocab:]], 1))


def train_C(Xc, Xe, Yt, train_pts, seed, interp_split=False, max_epochs=500, patience=30, batch=256):
    '''Xc: per-point counts (unscaled); Xe: per-point extra inputs (standardized here). Returns predict(Xc, Xe).'''
    tr = np.array(train_pts)
    if interp_split:                                     # Part I/II: fixed 20 % interpolation split, val = first 10 %
        p = np.random.RandomState(0).permutation(len(tr)); fit = tr[p[int(0.2 * len(tr)):]]
        nv = int(0.1 * len(fit)); val, fit = fit[:nv], fit[nv:]
    else:                                                # learning-curve style: seeded 10 % validation
        r0 = np.random.RandomState(seed); tr = tr.copy(); r0.shuffle(tr); nv = int(0.1 * len(tr)); val, fit = tr[:nv], tr[nv:]
    em, es = Xe[fit].mean(0), Xe[fit].std(0); es = np.where(es < 1e-6, 1.0, es)
    ym, ys = Yt[fit].mean(0), Yt[fit].std(0) + 1e-6
    X = torch.tensor(np.concatenate([Xc, (Xe - em) / es], 1).astype(np.float32)); Yn = torch.tensor((Yt - ym) / ys)
    torch.manual_seed(seed); rng = np.random.RandomState(seed)
    net = EmbedNet(Xc.shape[1], Xe.shape[1], Yt.shape[1]); opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5)
    L = nn.MSELoss(); best, st, bad = 1e9, None, 0; fit_t, val_t = torch.tensor(fit), torch.tensor(val)
    for ep in range(max_epochs):
        net.train(); perm = fit_t[torch.from_numpy(rng.permutation(len(fit)))]
        for s in range(0, len(perm), batch):
            i = perm[s:s + batch]; opt.zero_grad(); L(net(X[i]), Yn[i]).backward(); opt.step()
        net.eval()
        with torch.no_grad(): v = L(net(X[val_t]), Yn[val_t]).item()
        if v < best - 1e-5: best, st, bad = v, {k: t.clone() for k, t in net.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(st); net.eval()
    def predict(xc, xe):
        with torch.no_grad():
            return net(torch.tensor(np.concatenate([xc, (xe - em) / es], 1).astype(np.float32))).numpy() * ys + ym
    return predict, ep + 1


def pts_of(mask_mol, pm=None):
    pm = PM if pm is None else pm
    return np.where(mask_mol[pm])[0]

304400 labelled points (34s)


In [3]:
# ---- descriptors (R_desc) over the union main-group vocabulary, and generic trainers for R_count / R_count' / R_desc ----
MGS = sorted({sg.main_group(s) for s in VOCAB})
def desc(o):
    c, tq = np.zeros(len(MGS)), 0
    for s, q in o["subgroups"]:
        c[MGS.index(sg.main_group(int(s)))] += q; tq += q
    return [*(c / max(tq, 1)), o["oc"], o["nc"], np.log(o["M"]), np.log1p(tq)]
DESC = np.array([desc(o) for o in POOL], np.float32)

class Head(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.f = nn.Sequential(nn.Linear(n_in, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(),
                               nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 2))
    def forward(self, x): return self.f(x)

def train_std(Xm, Xe, Yt, train_pts, seed, unscaled_mol=False, interp_split=False, max_epochs=500, patience=30, batch=256):
    '''MLP on [molecular features | extras]; molecular features standardized (R_count, R_desc) or left raw (R_count').'''
    tr = np.array(train_pts)
    if interp_split:
        p = np.random.RandomState(0).permutation(len(tr)); fit = tr[p[int(0.2 * len(tr)):]]
        nv = int(0.1 * len(fit)); val, fit = fit[:nv], fit[nv:]
    else:
        r0 = np.random.RandomState(seed); tr = tr.copy(); r0.shuffle(tr); nv = int(0.1 * len(tr)); val, fit = tr[:nv], tr[nv:]
    X = np.concatenate([Xm, Xe], 1).astype(np.float32)
    xm, xs = X[fit].mean(0), X[fit].std(0); xs = np.where(xs < 1e-3, 1.0, xs)
    if unscaled_mol:
        xm[:Xm.shape[1]] = 0.0; xs[:Xm.shape[1]] = 1.0
    ym, ys = Yt[fit].mean(0), Yt[fit].std(0) + 1e-6
    Xn = torch.tensor((X - xm) / xs); Yn = torch.tensor((Yt - ym) / ys)
    torch.manual_seed(seed); rng = np.random.RandomState(seed)
    net = Head(X.shape[1]); opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5); L = nn.MSELoss()
    best, st, bad = 1e9, None, 0; fit_t, val_t = torch.tensor(fit), torch.tensor(val)
    for ep in range(max_epochs):
        net.train(); perm = fit_t[torch.from_numpy(rng.permutation(len(fit)))]
        for s in range(0, len(perm), batch):
            i = perm[s:s + batch]; opt.zero_grad(); L(net(Xn[i]), Yn[i]).backward(); opt.step()
        net.eval()
        with torch.no_grad(): v = L(net(Xn[val_t]), Yn[val_t]).item()
        if v < best - 1e-5: best, st, bad = v, {k: t.clone() for k, t in net.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(st); net.eval()
    def predict(xm_, xe_):
        with torch.no_grad():
            return net(torch.tensor(((np.concatenate([xm_, xe_], 1) - xm) / xs).astype(np.float32))).numpy() * ys + ym
    return predict, ep + 1

def fit_rep(rep, train_pts, seed, **kw):
    if rep == "R_emb":   return train_C(CNT[PM], EXTRA, Y, train_pts, seed, **kw)
    if rep == "R_count": return train_std(CNT[PM], EXTRA, Y, train_pts, seed, **kw)
    if rep == "R_count'": return train_std(CNT[PM], EXTRA, Y, train_pts, seed, unscaled_mol=True, **kw)
    if rep == "R_desc":  return train_std(DESC[PM], EXTRA, Y, train_pts, seed, **kw)

def feats_for(rep, idx):
    return (DESC if rep == "R_desc" else CNT)[PM[idx]]
REPS = ["R_count", "R_count'", "R_desc", "R_emb"]

## 1) Table 1, hash-split columns (Table 2 protocol, 5 seeds)

In [4]:
isB = SRC == "bimog"; isM = SRC == "mcm"
t1 = []
for pname, pmask in [("CHO", isB & (CAT == "cho")), ("BIMOG", isB)]:
    for rep in REPS:
        for seed in range(5):
            pred, ep = fit_rep(rep, pts_of(pmask & ~TEST), seed, interp_split=True)
            g = pts_of(pmask & TEST); e = np.abs(pred(feats_for(rep, g), EXTRA[g]) - Y[g])
            t1.append({"pool": pname, "rep": rep, "seed": seed, "gen_w": e[:, 0].mean(), "gen_org": e[:, 1].mean()})
        print(f"  {pname} {rep} done ({time.time()-T0:.0f}s)")
t1 = pd.DataFrame(t1); t1.to_csv("U_table1_hash_runs.csv", index=False)
display(t1.groupby(["pool", "rep"], sort=False)[["gen_w", "gen_org"]].agg(fmt))

  CHO R_count done (58s)


  CHO R_count' done (84s)


  CHO R_desc done (111s)


  CHO R_emb done (144s)


  BIMOG R_count done (186s)


  BIMOG R_count' done (217s)


  BIMOG R_desc done (256s)


  BIMOG R_emb done (306s)


gen_w        gen_org
pool  rep                                   
CHO   R_count   0.049 ± 0.008  0.494 ± 0.064
      R_count'  0.031 ± 0.001  0.245 ± 0.031
      R_desc    0.022 ± 0.002  0.246 ± 0.016
      R_emb     0.022 ± 0.002  0.181 ± 0.035
BIMOG R_count   0.032 ± 0.005  0.384 ± 0.023
      R_count'  0.026 ± 0.002  0.247 ± 0.018
      R_desc    0.025 ± 0.002  0.315 ± 0.021
      R_emb     0.015 ± 0.001  0.147 ± 0.016

## 2) Table 1, CHO five-fold cross-validation (same labels, 3 seeds)

In [5]:
cho_ids = np.where(isB & (CAT == "cho"))[0]; fold = H % 5
cv = []
for rep in REPS:
    for seed in range(3):
        per = {}
        for k in range(5):
            trm = np.zeros(len(POOL), bool); trm[cho_ids[fold[cho_ids] != k]] = True
            tem = np.zeros(len(POOL), bool); tem[cho_ids[fold[cho_ids] == k]] = True
            pred, _ = fit_rep(rep, pts_of(trm), 1000 * seed + k, interp_split=True)
            g = pts_of(tem); e = np.abs(pred(feats_for(rep, g), EXTRA[g]) - Y[g])[:, 1]
            for m, v in pd.Series(e).groupby(PM[g]).mean().items(): per[m] = v
        cv.append({"rep": rep, "seed": seed, "oof_org": np.mean(list(per.values())), **{f"m{m}": v for m, v in per.items()}})
    print(f"  CV {rep} done ({time.time()-T0:.0f}s)")
cv = pd.DataFrame(cv); cv.to_csv("U_table1_cv_runs.csv", index=False)
display(cv.groupby("rep", sort=False)["oof_org"].agg(fmt))
mcols = [c for c in cv.columns if c.startswith("m")]
mm = cv.groupby("rep")[mcols].mean().T
for a, b in [("R_emb", "R_desc"), ("R_desc", "R_count"), ("R_count'", "R_desc")]:
    d = mm[a] - mm[b]
    print(f"{a} vs {b}: mean Δ {d.mean():+.3f} ({100*d.mean()/mm[b].mean():+.0f}%), {a} better for {(d<0).mean()*100:.0f}% of molecules, "
          f"Wilcoxon p = {stats.wilcoxon(mm[a], mm[b]).pvalue:.2g}")

  CV R_count done (378s)


  CV R_count' done (446s)


  CV R_desc done (533s)


  CV R_emb done (628s)


rep
R_count     0.352 ± 0.015
R_count'    0.262 ± 0.003
R_desc      0.302 ± 0.007
R_emb       0.174 ± 0.011
Name: oof_org, dtype: str

R_emb vs R_desc: mean Δ -0.128 (-42%), R_emb better for 65% of molecules, Wilcoxon p = 5.2e-09
R_desc vs R_count: mean Δ -0.050 (-14%), R_desc better for 58% of molecules, Wilcoxon p = 0.0017
R_count' vs R_desc: mean Δ -0.040 (-13%), R_count' better for 42% of molecules, Wilcoxon p = 0.62


## 3) Table 4: R_desc MLP and GNN on the same MCM subsets and labels (3 seeds)

In [6]:
# ---- GNN (Part II architecture, plain PyTorch) on the shared label set ----
ATOMS = ["C", "O", "N", "Cl", "Br", "F", "I", "S"]
BONDS = [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE, Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]
def graph(smi):
    mol = Chem.MolFromSmiles(smi)
    x = [[float(a.GetSymbol() == s) for s in ATOMS] + [a.GetDegree() / 4, a.GetTotalNumHs() / 4, float(a.GetIsAromatic()),
         float(a.IsInRing()), a.GetFormalCharge()] for a in mol.GetAtoms()]
    ei, ea = [], []
    for b in mol.GetBonds():
        f = [float(b.GetBondType() == t) for t in BONDS]; i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        ei += [(i, j), (j, i)]; ea += [f, f]
    if not ei: ei, ea = [(0, 0)], [[0.0] * 4]
    return np.array(x, np.float32), np.array(ei, np.int64).T, np.array(ea, np.float32)
GR = [graph(o["smiles"]) for o in POOL]

class Bank:
    def __init__(self, ids):
        xs, eis, eas, own, off = [], [], [], [], 0
        for k, m in enumerate(ids):
            x, ei, ea = GR[m]; xs.append(x); eis.append(ei + off); eas.append(ea); own.append(np.full(len(x), k)); off += len(x)
        self.x = torch.tensor(np.concatenate(xs)); self.ei = torch.tensor(np.concatenate(eis, 1))
        self.ea = torch.tensor(np.concatenate(eas)); self.own = torch.tensor(np.concatenate(own)); self.n = len(ids)
        self.loc = -np.ones(len(POOL), int); self.loc[np.array(ids)] = np.arange(len(ids))

class GINE(nn.Module):
    def __init__(self, h):
        super().__init__(); self.mlp = nn.Sequential(nn.Linear(h, h), nn.ReLU(), nn.Linear(h, h))
    def forward(self, h, ei, e):
        return self.mlp(h + torch.zeros_like(h).index_add_(0, ei[1], torch.relu(h[ei[0]] + e)))

class GNN(nn.Module):
    def __init__(self, h=64, emb=64, L=3):
        super().__init__()
        self.ap = nn.Linear(len(ATOMS) + 5, h); self.ep = nn.ModuleList([nn.Linear(4, h) for _ in range(L)])
        self.cv = nn.ModuleList([GINE(h) for _ in range(L)]); self.bn = nn.ModuleList([nn.BatchNorm1d(h) for _ in range(L)])
        self.out = nn.Linear(2 * h, emb)
        self.head = nn.Sequential(nn.Linear(emb + 2, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 2))
    def embed(self, b):
        h = self.ap(b.x)
        for c, n, e in zip(self.cv, self.bn, self.ep): h = torch.relu(n(c(h, b.ei, e(b.ea))))
        s = torch.zeros(b.n, h.shape[1]).index_add_(0, b.own, h)
        cnt = torch.zeros(b.n).index_add_(0, b.own, torch.ones(len(h))).clamp(min=1)
        return self.out(torch.cat([s / cnt[:, None], s], 1))

def train_gnn(train_ids, seed, eval_masks, max_epochs=40, patience=6, batch=1024):
    rng = np.random.RandomState(seed); torch.manual_seed(seed)
    m = np.zeros(len(POOL), bool); m[train_ids] = True; tr = pts_of(m); rng.shuffle(tr)
    nv = int(0.1 * len(tr)); val, fit = tr[:nv], tr[nv:]
    em, es = EXTRA[fit].mean(0), EXTRA[fit].std(0); ym, ys = Y[fit].mean(0), Y[fit].std(0) + 1e-6
    C_ = torch.tensor((EXTRA - em) / es); Yn = torch.tensor((Y - ym) / ys); P_ = PM
    bank = Bank(list(train_ids)); net = GNN(); opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5)
    L = nn.MSELoss(); best, st, bad = 1e9, None, 0
    def fwd(b, idx):
        return net.head(torch.cat([net.embed(b)[torch.tensor(b.loc[P_[idx]])], C_[idx]], 1))
    for ep in range(max_epochs):
        net.train(); perm = fit[rng.permutation(len(fit))]
        for s in range(0, len(perm), batch):
            idx = perm[s:s + batch]; opt.zero_grad(); L(fwd(bank, idx), Yn[idx]).backward(); opt.step()
        net.eval()
        with torch.no_grad(): v = L(fwd(bank, val), Yn[val]).item()
        if v < best - 1e-5: best, st, bad = v, {k: t.clone() for k, t in net.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(st); net.eval(); out = {}
    for name, em_ in eval_masks.items():
        ids = np.where(em_)[0]; b = Bank(list(ids)); g = pts_of(em_)
        with torch.no_grad(): p = fwd(b, g).numpy() * ys + ym
        out[name] = np.abs(p - Y[g]).mean(0)
    return out, ep + 1

mcm_tr = np.where(isM & ~TEST)[0]; evals = {"MCM": isM & TEST, "BIMOG": isB & TEST}
t4 = []
for seed in range(3):
    order = np.random.RandomState(100 + seed).permutation(mcm_tr)           # same nested subsets as the R_emb rows
    for n in (400, len(order)):
        sub = order[:n]; m = np.zeros(len(POOL), bool); m[sub] = True
        pred, ep = fit_rep("R_desc", pts_of(m), seed, max_epochs=300)
        rec = {"model": "R_desc MLP", "n": n, "seed": seed}
        for name, em_ in evals.items():
            g = pts_of(em_); e = np.abs(pred(DESC[PM[g]], EXTRA[g]) - Y[g]); rec[f"{name} org"] = e[:, 1].mean(); rec[f"{name} w"] = e[:, 0].mean()
        t4.append(rec)
        res, ep = train_gnn(sub, seed, evals)
        t4.append({"model": "GNN", "n": n, "seed": seed, **{f"{k} org": v[1] for k, v in res.items()}, **{f"{k} w": v[0] for k, v in res.items()}})
        print(f"  seed {seed} n={n}: R_desc MCM {rec['MCM org']:.3f} | GNN MCM {res['MCM'][1]:.3f} ({time.time()-T0:.0f}s)")
t4 = pd.DataFrame(t4); t4.to_csv("U_table4_desc_gnn_runs.csv", index=False)
display(t4.groupby(["model", "n"])[["MCM org", "BIMOG org", "MCM w"]].agg(fmt))

  seed 0 n=400: R_desc MCM 0.167 | GNN MCM 0.208 (664s)


  seed 0 n=2855: R_desc MCM 0.097 | GNN MCM 0.087 (1282s)


  seed 1 n=400: R_desc MCM 0.157 | GNN MCM 0.193 (1325s)


  seed 1 n=2855: R_desc MCM 0.099 | GNN MCM 0.088 (1924s)


  seed 2 n=400: R_desc MCM 0.171 | GNN MCM 0.201 (1968s)


  seed 2 n=2855: R_desc MCM 0.098 | GNN MCM 0.084 (2921s)


MCM org      BIMOG org          MCM w
model      n                                                
GNN        400   0.200 ± 0.007  1.732 ± 0.242  0.029 ± 0.001
           2855  0.086 ± 0.002  1.325 ± 0.121  0.013 ± 0.002
R_desc MLP 400   0.165 ± 0.007  1.561 ± 0.040  0.021 ± 0.002
           2855  0.098 ± 0.001  1.260 ± 0.126  0.013 ± 0.000

## 4) Release checkpoint: R_emb ensemble trained on all BIMOG + MCM molecules (3 seeds)

In [7]:
import os, json as _json
os.makedirs("checkpoints", exist_ok=True)
allm = isB | isM
members = []
for seed in range(3):
    tr = pts_of(allm); r0 = np.random.RandomState(seed); tr = tr.copy(); r0.shuffle(tr); nv = int(0.1 * len(tr)); val, fit = tr[:nv], tr[nv:]
    em, es = EXTRA[fit].mean(0), EXTRA[fit].std(0); ym, ys = Y[fit].mean(0), Y[fit].std(0) + 1e-6
    X = torch.tensor(np.concatenate([CNT[PM], (EXTRA - em) / es], 1).astype(np.float32)); Yn = torch.tensor((Y - ym) / ys)
    torch.manual_seed(seed); rng = np.random.RandomState(seed)
    net = EmbedNet(len(VOCAB), 2); opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5); L = nn.MSELoss()
    best, st, bad = 1e9, None, 0; fit_t, val_t = torch.tensor(fit), torch.tensor(val)
    for ep in range(300):
        net.train(); perm = fit_t[torch.from_numpy(rng.permutation(len(fit)))]
        for s in range(0, len(perm), 256):
            i = perm[s:s + 256]; opt.zero_grad(); L(net(X[i]), Yn[i]).backward(); opt.step()
        net.eval()
        with torch.no_grad(): v = L(net(X[val_t]), Yn[val_t]).item()
        if v < best - 1e-5: best, st, bad = v, {k: t.clone() for k, t in net.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= 30: break
    torch.save(st, f"checkpoints/remb_bimog_mcm_seed{seed}.pt")
    members.append({"file": f"remb_bimog_mcm_seed{seed}.pt", "seed": seed, "epochs": ep + 1, "val_mse_std": best,
                    "extra_mean": em.tolist(), "extra_std": es.tolist(), "y_mean": ym.tolist(), "y_std": ys.tolist()})
    print(f"  member seed {seed}: {ep+1} epochs, val MSE (standardized) {best:.4f} ({time.time()-T0:.0f}s)")
meta = {"description": "AIOMFAC binary water-organic surrogate, representation R_emb (learned subgroup embedding), "
                       "trained on all BIMOG (368) + MCM v3.3.1 (3337) molecules; ensemble of 3 seeds",
        "aiomfac_py_version": __import__("aiomfac_py").__version__, "subgroup_vocabulary": VOCAB,
        "inputs": ["subgroup counts (unscaled, ordered as subgroup_vocabulary)", "w_water (mass fraction)", "T_norm = (T - 293.15)/20"],
        "outputs": ["ln_gamma_water", "ln_gamma_organic"], "architecture": {"embedding_dim": 16, "hidden": [128, 128, 128], "activation": "ReLU"},
        "training_ranges": {"w_water": [0.02, 0.98], "T_K": [273.15, 313.15]}, "n_molecules": int(allm.sum()), "members": members}
_json.dump(meta, open("checkpoints/remb_bimog_mcm_meta.json", "w"), indent=1)
print("saved checkpoints/remb_bimog_mcm_*.pt and meta json; total runtime", f"{time.time()-T0:.0f}s")

  member seed 0: 87 epochs, val MSE (standardized) 0.0005 (3002s)


  member seed 1: 102 epochs, val MSE (standardized) 0.0006 (3095s)


  member seed 2: 89 epochs, val MSE (standardized) 0.0006 (3170s)
saved checkpoints/remb_bimog_mcm_*.pt and meta json; total runtime 3170s
